# **5CS037 – Concepts and Technologies of AI**  
## **Worksheet 10: Naive Bayes and Feature Selection**

---

This notebook contains:

- **Exercise 1:** Implementation of Naive Bayes Algorithm  
- **Exercise 2:** Feature Selection using Wrapper Methods (RFE)



## **Exercise 1 – Implementation of Naive Bayes Algorithm**  
### **Sentiment Analysis using IMDB Movie Review Dataset**

---


### **Part 1 – Data Loading and Preprocessing**

**Tasks performed:**

- Convert all reviews to **lowercase**
- Remove **non-alphabetic characters**
- **Tokenize** text and remove **stopwords**
- Apply **stemming** to reduce words to root form


In [2]:
from google.colab import files
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [3]:
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [4]:
data = pd.read_csv("IMDB Dataset.csv")

nltk.download('stopwords')
stemmer = PorterStemmer()
stops = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z ]', '', text)
    tokens = [stemmer.stem(w) for w in text.split() if w not in stops]
    return " ".join(tokens)

data['review'] = data['review'].apply(preprocess)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### **Part 2 – Train-Test Split and Model Training**

**Performed Steps:**

- Split the dataset into **80% training** and **20% testing**
- Apply **Bag-of-Words** using `CountVectorizer`
- Train a **Multinomial Naive Bayes** classifier


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    data['review'], data['sentiment'], test_size=0.2, random_state=42
)

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

### **Part 3 – Model Evaluation**

**Evaluation Metrics used:**

- **Accuracy**
- **Precision, Recall, F1-score**
- **Confusion Matrix**
- **ROC-AUC Score**


In [6]:
y_pred = model.predict(X_test_vec)
y_prob = model.predict_proba(X_test_vec)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score((y_test == 'positive').astype(int), y_prob))

Accuracy: 0.8554
              precision    recall  f1-score   support

    negative       0.84      0.87      0.86      4961
    positive       0.87      0.84      0.85      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

Confusion Matrix:
 [[4322  639]
 [ 807 4232]]
ROC-AUC: 0.9224499818568961


## **Exercise 2 – Feature Selection using Wrapper Methods**  
### **Breast Cancer Prognostic Dataset**

---


### **Part 1 – Data Loading and Preprocessing**

**Tasks performed:**

- Load the Breast Cancer dataset
- Review **summary statistics**
- Verify **missing values**
- Split data into **80% training** and **20% testing**


In [7]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [8]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### **Part 2 – Recursive Feature Elimination (RFE)**

**Method:**

- Wrapper-based feature selection using **Logistic Regression**
- Selection of **Top 5 most important features**


In [9]:
log_reg = LogisticRegression(max_iter=500)
rfe = RFE(log_reg, n_features_to_select=5)
rfe.fit(X_train, y_train)

selected_features = X_train.columns[rfe.support_]
selected_features

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Index(['mean radius', 'texture error', 'worst radius', 'worst compactness',
       'worst concavity'],
      dtype='object')

### **Part 3 – Model Training and Evaluation**

**Metrics computed:**

- **Accuracy**
- **Precision**
- **Recall**
- **F1-Score**
- **ROC-AUC**


In [10]:
log_reg.fit(X_train[selected_features], y_train)

y_pred = log_reg.predict(X_test[selected_features])
y_prob = log_reg.predict_proba(X_test[selected_features])[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9736842105263158
Precision: 0.9722222222222222
Recall: 0.9859154929577465
F1-Score: 0.9790209790209791
ROC-AUC: 0.9983622666229938


### **Part 4 – Experiment: Effect of Number of Features**

**Objective:**

- Evaluate model performance using **different numbers of selected features**
- Compare results for:
  - **Top 3 features**
  - **Top 5 features**
  - **Top 7 features**


In [11]:
results = {}

for n_features in [3, 5, 7]:
    rfe = RFE(LogisticRegression(max_iter=500), n_features_to_select=n_features)
    rfe.fit(X_train, y_train)

    selected = X_train.columns[rfe.support_]

    model = LogisticRegression(max_iter=500)
    model.fit(X_train[selected], y_train)

    y_pred = model.predict(X_test[selected])
    y_prob = model.predict_proba(X_test[selected])[:, 1]

    results[n_features] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

pd.DataFrame(results).T

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

,Accuracy,Precision,Recall,F1-Score,ROC-AUC
3,0.807018,0.781609,0.957746,0.860759,0.937439
5,0.973684,0.972222,0.985915,0.979021,0.998362
7,0.973684,0.972222,0.985915,0.979021,0.998362


**Observation:**

- Feature selection impacts model performance.
- Using too few features may reduce predictive power.
- Selecting an optimal number of features helps balance **model simplicity** and **performance**.
